In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from tqdm.notebook import tqdm
from src.utils import make_pattern, section_manager, clean_text
import en_core_sci_sm
from pathlib import Path
import json

tqdm.pandas()
load_dotenv()
nlp_core = en_core_sci_sm.load()

Szükséges mappastruktúra létehozása (Create required folder structure)

In [ ]:
base_dir = Path("../data/processed")
base_dir.mkdir(parents=True, exist_ok=True)

train_dirs = {
    "without_drop_frequent": base_dir / "frequent_chapter" / "without_dropped_sections",
    "with_drop_frequent": base_dir / "frequent_chapter" / "with_dropped_sections",
    "without_drop_top50": base_dir / "top_50_code" / "without_dropped_sections",
    "with_drop_top50": base_dir / "top_50_code" / "with_dropped_sections"
}

for path in train_dirs.values():
    path.mkdir(parents=True, exist_ok=True)
    
description_dir = Path("../data/descriptions")
description_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
diagnoses_icd = pd.read_csv(os.getenv("DIAGNOSES_ICD"))
print("\ndiagnoses_icd:\n")
diagnoses_icd.info(memory_usage="deep", show_counts=True)

d_icd_diagnoses = pd.read_csv(os.getenv("D_ICD_DIAGNOSES"))
print("\nd_icd_diagnoses:\n")
d_icd_diagnoses.info(memory_usage="deep", show_counts=True)

discharge = pd.read_csv(os.getenv("DISCHARGE"))
print("\ndischarge:\n")
discharge.info(memory_usage="deep", show_counts=True)

### hadm_id-val nem rendelkező oszlopok eltávolítása (Remove rows with missing hadm_id)

In [ ]:
diagnoses_icd_len = len(diagnoses_icd)

diagnoses_icd.dropna(subset=["hadm_id"], inplace=True)

print(f"Rows dropped: {diagnoses_icd_len-len(diagnoses_icd)}")
print(f"Remaining rows: {len(diagnoses_icd)}")

### Szükséges oszlopok megtartása (Keep required columns) 

In [ ]:
diagnoses_icd = diagnoses_icd[["subject_id", "hadm_id", "icd_code", "icd_version"]]

### Az összes olyan hadm_id-val rendelkező sor eldobása, ahol az adott azonosítóhoz tartozó bármelyik sorban hiányzó adat található (Drop all rows associated with a hadm_id if any record for that identifier contains missing values)

In [ ]:
diagnoses_icd_len = len(diagnoses_icd)

invalid_hadm_ids = diagnoses_icd[diagnoses_icd.isna().any(axis=1)]["hadm_id"].unique()

diagnoses_icd = diagnoses_icd[~diagnoses_icd["hadm_id"].isin(invalid_hadm_ids)]

print(f"Removed hadm_id count: {len(invalid_hadm_ids)}")
print(f"Rows dropped: {diagnoses_icd_len-len(diagnoses_icd)}")
print(f"Remaining rows: {len(diagnoses_icd)}")

### Az összes olyan hadm_id-val rendelkező sor megtartása amihez csak 10-es verziójú kód tartozik (Retain rows where the hadm_id is exclusively associated with version 10 codes)

In [ ]:
v9_hadm_ids = set(diagnoses_icd[diagnoses_icd["icd_version"] == 9]["hadm_id"])
v10_hadm_ids = set(diagnoses_icd[diagnoses_icd["icd_version"] == 10]["hadm_id"])

only_v9_hadm_ids = v9_hadm_ids - v10_hadm_ids
only_v10_hadm_ids = v10_hadm_ids - v9_hadm_ids
both_v_hadm_ids = v9_hadm_ids & v10_hadm_ids

print(f"Only ICD-9:  {len(only_v9_hadm_ids)}") 
print(f"Only ICD-10: {len(only_v10_hadm_ids)}") 
print(f"Both versions: {len(both_v_hadm_ids)}")

In [ ]:
diagnoses_icd_len = len(diagnoses_icd)

diagnoses_icd = diagnoses_icd[diagnoses_icd["hadm_id"].isin(only_v10_hadm_ids)]

print(f"Removed hadm_id count: {len(only_v9_hadm_ids | both_v_hadm_ids)}")
print(f"Rows dropped: {diagnoses_icd_len-len(diagnoses_icd)}")
print("Remaining rows:", len(diagnoses_icd))

#### Az "icd_version" oszlop eldobása (Drop the "icd_version" column)

In [ ]:
diagnoses_icd = diagnoses_icd.drop(columns=["icd_version"])

### Az összes olyan hadm_id-val rendelkező sor eldobása amihez rossz formátumú kód tartozik (Drop all rows associated with hadm_ids containing malformed codes)

In [ ]:
diagnoses_icd_len = len(diagnoses_icd)

diagnoses_icd["icd_code"] = diagnoses_icd["icd_code"].str.upper()

format_pattern = r"^[A-Z][0-9][A-Z0-9]{1,5}$"

invalid_rows = diagnoses_icd[~diagnoses_icd["icd_code"].str.match(format_pattern)]

invalid_hadm_ids = invalid_rows["hadm_id"].unique()

diagnoses_icd = diagnoses_icd[~diagnoses_icd["hadm_id"].isin(invalid_hadm_ids)]

diagnoses_icd = diagnoses_icd[diagnoses_icd["icd_code"].str.match(format_pattern)]

print(f"Removed hadm_id count: {len(invalid_hadm_ids)}")
print(f"Rows dropped: {diagnoses_icd_len - len(diagnoses_icd)}")
print(f"Remaining rows: {len(diagnoses_icd)}")

### Chapter-ek meghatározása a kódokhoz (Defining chapters for the code)
<https://ftp.cdc.gov/pub/health_statistics/nchs/publications/ICD10CM/2026/ICD-10-CM-October-2025-Guidelines.pdf>

In [ ]:
icd_10_cm_chapters = [
    {"chapter": "A00-B99", "long_title": "Certain Infectious and Parasitic Diseases"},
    {"chapter": "C00-D49", "long_title": "Neoplasms"},
    {"chapter": "D50-D89", "long_title": "Disease of the blood and blood-forming organs and certain disorders involving the immune mechanism"},
    {"chapter": "E00-E89", "long_title": "Endocrine, Nutritional, and Metabolic Diseases"},
    {"chapter": "F01-F99", "long_title": "Mental, Behavioral and Neurodevelopmental disorders"},
    {"chapter": "G00-G99", "long_title": "Diseases of the Nervous System"},
    {"chapter": "H00-H59", "long_title": "Diseases of the Eye and Adnexa"},
    {"chapter": "H60-H95", "long_title": "Diseases of the Ear and Mastoid Process"},
    {"chapter": "I00-I99", "long_title": "Diseases of the Circulatory System"},
    {"chapter": "J00-J99", "long_title": "Diseases of the Respiratory System"},
    {"chapter": "K00-K95", "long_title": "Diseases of the Digestive System"},
    {"chapter": "L00-L99", "long_title": "Diseases of the Skin and Subcutaneous Tissue"},
    {"chapter": "M00-M99", "long_title": "Diseases of the Musculoskeletal System and Connective Tissue"},
    {"chapter": "N00-N99", "long_title": "Diseases of Genitourinary System"},
    {"chapter": "O00-O9A", "long_title": "Pregnancy, Childbirth, and the Puerperium"},
    {"chapter": "P00-P96", "long_title": "Certain Conditions Originating in the Perinatal Period"},
    {"chapter": "Q00-QA1", "long_title": "Congenital malformations, deformations, and chromosomal abnormalities"},
    {"chapter": "R00-R99", "long_title": "Symptoms, signs, and abnormal clinical and laboratory findings, not elsewhere classified"},
    {"chapter": "S00-T88", "long_title": "Injury, poisoning, and certain other consequences of external causes"},
    {"chapter": "V00-Y99", "long_title": "External Causes of Morbidity"},
    {"chapter": "Z00-Z99", "long_title": "Factors influencing health status and contact with health services"},
    {"chapter": "U00-U85", "long_title": "Codes for Special Purposes"},
    {"chapter": "Unknown", "long_title": "Unknown"}
]

In [ ]:
def get_chapter_for_code(code, chapters):
    for chapter in chapters:
        if chapter["chapter"] == "Unknown":
            continue
        start, end = chapter["chapter"].split("-")
        if start <= code[:3] <= end:
                return chapter
    return "Unknown"

In [ ]:
diagnoses_icd["chapter"] = diagnoses_icd["icd_code"].progress_apply(
    lambda x: get_chapter_for_code(x, icd_10_cm_chapters)
)

In [ ]:
icd_10_cm_chapters = pd.DataFrame(icd_10_cm_chapters)

icd_10_cm_chapters.to_json(
    description_dir / "icd_chapter_descriptions.json", 
    orient="records",   
    indent=4,           
    force_ascii=False   
)

### Az összes olyan hadm_id-val rendelkező sor eldobása aminél nem sikerült chapter-t azonosítani (Drop all rows associated with hadm_ids where chapter assignment failed)

In [ ]:
unknown_codes = diagnoses_icd[diagnoses_icd["chapter"] == "Unknown"]["icd_code"].value_counts()

if not unknown_codes.empty:
    display(unknown_codes)
else:
    print("All codes were successfully identified\n")
    
diagnoses_icd_len = len(diagnoses_icd)
    
invalid_hadm_ids = diagnoses_icd[diagnoses_icd["chapter"] == "Unknown"]["hadm_id"].unique()
diagnoses_icd = diagnoses_icd[~diagnoses_icd["hadm_id"].isin(invalid_hadm_ids)]
    
print(f"Removed hadm_id count: {len(invalid_hadm_ids)}")
print(f"Rows dropped: {diagnoses_icd_len - len(diagnoses_icd)}")
print(f"Remaining rows: {len(diagnoses_icd)}")


#### Nem reprezentált chapter-ek (Unrepresented chapters)

In [ ]:
unrepresented_chapters = set(icd_10_cm_chapters) - set(diagnoses_icd["chapter"].unique())
unrepresented_chapters.discard('Unknown')
print(f"Unrepresented chapters: {unrepresented_chapters or 'None'}")

represented_chapters = diagnoses_icd['chapter'].unique()
print(f"Represented chapters: {sorted(represented_chapters)}")

print(f"Number of represented chapters: {len(represented_chapters)}")

### Adatok aggregálása (Data aggregation)

In [ ]:
diagnoses_icd = diagnoses_icd.groupby(["hadm_id", "subject_id"]).agg({
    "icd_code": lambda x: sorted(set(x)),             
    "chapter": lambda x: sorted(set(x))
}).reset_index()

print(f"Rows after aggregation: {len(diagnoses_icd)}")

### Zárójelentések (DS) szűrése és a hiányzó azonosítók eltávolítása (Filtering Discharge Summaries (DS) and removing missing identifiers)

In [ ]:
discharge_len = len(discharge)

discharge = discharge[discharge["note_type"] == "DS"]
discharge = discharge.dropna(subset=['hadm_id', 'subject_id'])

print(f"Rows dropped: {discharge_len - len(discharge)}")
print(f"Remaining rows: {len(discharge)}")

### Szükséges oszlopok megtartása (Keep required columns) 

In [ ]:
discharge = discharge[['subject_id', 'hadm_id', 'text']]

### Minden olyan sor eldobása, ami nem egyedi hadm_id-hez tartozik (Drop all rows with non-unique hadm_ids)

In [ ]:
discharge_len = len(discharge)

discharge = discharge.drop_duplicates(subset=['hadm_id'], keep=False)

print(f"Rows dropped: {discharge_len - len(discharge)}")
print(f"Remaining rows: {len(discharge)}")

## A diagnoses_icd és a discharge tábla összefésülése (Merging the diagnoses_icd and discharge tables)

In [ ]:
dataset = pd.merge(discharge, diagnoses_icd, on=["subject_id", "hadm_id"], how="inner")

print(f"diagnoses_icd rows: {len(diagnoses_icd)}")
print(f"discharge rows: {len(discharge)}")
print(f"base_dataset rows: {len(dataset)}")

In [ ]:
del diagnoses_icd
del discharge

##### Adathalmaz elmentése analizálásra (Saving the dataset for analysis)

In [ ]:
dataset.to_parquet(base_dir / "base_dataset.parquet", engine="pyarrow", index=False)

# Tanító adathalmazok létrehozása (Creating training datasets) 

Tanító adathalmazok (Training datasets):
- Gyakori ICD-10-CM chapter osztályozás (Frequent ICD-10-CM chapter classification):
    - Nyers zárójelentéseket tartalmazó (Dataset containing raw discharge summaries)
    - Tisztított zárójelentéseket tartalmazó (Dataset containing cleaned discharge summaries)
    - A zárójeléntésből csak adott szekciókat tartalmazó (Dataset ontaining only specific sections of the discharge summary)
    - A zárójeléntésből csak adott, tisztított szekciókat tartalmazó (Dataset containing only specific, cleaned sections of the discharge summary)
- Top 50 ICD-10-CM kód osztályozás (Top 50 ICD-10-CM code classification):
    - Nyers zárójelentéseket tartalmazó (Dataset containing raw discharge summaries)
    - Tisztított zárójelentéseket tartalmazó (Dataset containing cleaned discharge summaries)
    - A zárójeléntésből csak adott szekciókat tartalmazó (Dataset ontaining only specific sections of the discharge summary)
    - A zárójeléntésből csak adott, tisztított szekciókat tartalmazó (Dataset containing only specific, cleaned sections of the discharge summary)

Az öt célszekció(The four target sections):
   - Főpanasz (Chief Complaint)
   - Jelen betegség kórfejlődése (History of Present Illness)
   - Kórelőzmény (Past Medical History)
   - Rövid kórházi összefoglaló (Brief Hospital Course)
   - Záródiagnózis (Discharge Diagnosis)

Az adatszivárgási kockázatok elkerülése érdekében két külön adathalmaz variáns kerül előállításra: az egyik tartalmazza a 'Záródiagnózis' szekciót, a másikból pedig az kihagyásra kerül (To avoid data leakage risks, two separate dataset variants will be created: one including the 'Discharge Diagnosis' section, and another from which it is excluded) 

Az összehasonlíthatóság érdekében az összes adathalmaz a legszűkebb részhalmazhoz lesz igazítva. Ennek megfelelően a nyers és a tisztított teljes zárójelentések tartalmazó adatsorok közül is csak azok az esetek lesznek megtartva, ahol a célszekciók azonosítása sikeres volt.
(To ensure comparability, all datasets will be aligned to the most restrictive subset. Therefore, from the datasets containing raw and cleaned full discharge summaries, only those cases where identification of target sections were successful will be retained.)

#### Szövegtisztítás futtatási paraméterei (Text cleaning runtime parameters)

In [ ]:
BATCH_SIZE = 64
N_PROCESS = 6

### Szűrés azokra az adatsorokra ahol az öt célszekció jól elkülöníthető (Filtering for records where the five target sections are clearly identifiable.)

In [ ]:
with open("../t_s_config.json", "r", encoding="utf-8") as f:
    t_s_config_data = json.load(f)

print(json.dumps(t_s_config_data, indent=4, ensure_ascii=False))

In [ ]:
dataset_len = len(dataset)

section_patterns = {
    key: [make_pattern(start_s), make_pattern(end_s)] 
    for key, (start_s, end_s) in t_s_config_data["section_flags"].items()
}

dataset = dataset[dataset["text"].progress_apply(lambda x: section_manager(x, section_patterns))]

print(f"Rows dropped: {dataset_len - len(dataset)}")
print(f"Remaining rows: {len(dataset)}")

##### Adathalmaz elmentése analizálásra (Saving the dataset for analysis)

In [ ]:
dataset.to_parquet(base_dir / "section_filtered_dataset.parquet", engine="pyarrow", index=False)

### Alapadathalmazok létrehozása tanításra és analizálásra (Creating and saving the base datasets for analysis and training)

In [ ]:
frequent_chapter_dataset = dataset.copy()
top_50_code_dataset = dataset.copy() 

#### Gyakori ICD-10-CM chapter (Frequent ICD-10-CM chapter)

In [ ]:
frequent_chapter_dataset.drop(columns=["icd_code"], inplace=True)

frequent_chapter_dataset_len = len(frequent_chapter_dataset)

chapter_counts = frequent_chapter_dataset["chapter"].explode().value_counts()

print(f"Chapter counts:\n{chapter_counts}\n")

threshold = 0.01 * len(frequent_chapter_dataset)
frequent_chapter_counts = chapter_counts[chapter_counts >= threshold]

print(f"Threshold: {threshold}\n")
print(f"Chapter counts after threshold:\n{frequent_chapter_counts}\n")

chapter_set = set(chapter_counts.index)
frequent_chapter_set = set(frequent_chapter_counts.index)

frequent_chapter_dataset["chapter"] = frequent_chapter_dataset["chapter"].apply(
    lambda chapter_list: [chapter for chapter in chapter_list if chapter in frequent_chapter_set]
)

print(f"Removed chapters:\n{chapter_set - frequent_chapter_set}\n")

frequent_chapter_dataset = frequent_chapter_dataset[frequent_chapter_dataset["chapter"].str.len() > 0].reset_index(drop=True)

print(f"Rows dropped in 'frequent_chapter_base_dataset': {frequent_chapter_dataset_len-len(frequent_chapter_dataset)}")
print(f"Remaining rows in 'frequent_chapter_base_dataset': {len(frequent_chapter_dataset)}")

frequent_chapter_dataset.to_parquet(train_dirs["without_drop_frequent"] / "base_dataset.parquet", engine="pyarrow", index=False)

###### Tisztított zárójelentéseket tartalmazó adathalmazok létrehozása majd elmentése analízisre és tanításra (Creating and saving datasets containing cleaned discharge summaries for analysis and training)

In [ ]:
dataset = frequent_chapter_dataset.copy()
dataset["text"] = clean_text(dataset["text"].tolist(), "tfidf", nlp_core, BATCH_SIZE, N_PROCESS)
dataset.to_parquet(train_dirs["without_drop_frequent"] / "tfidf_cleaned_base_dataset.parquet", engine="pyarrow", index=False)

In [ ]:
dataset = frequent_chapter_dataset.copy()
dataset["text"] = clean_text(dataset["text"].tolist(), "modernbert")
dataset.to_parquet(train_dirs["without_drop_frequent"] / "modernbert_cleaned_base_dataset.parquet", engine="pyarrow", index=False)

#### Záródiagnózisokat nem tartalmazó adathalmazok létrehozása majd elmentése analízisre és tanításra (Creating and saving datasets excluding discharge diagnoses for analysis and training)

In [ ]:
dataset = frequent_chapter_dataset.copy()
dataset["text"] = dataset["text"].progress_apply(
    lambda x: section_manager(
        x, 
        section_patterns, 
        "drop", 
        t_s_config_data["with_dropped_sections"]["sections_to_drop"]))
dataset.to_parquet(train_dirs["with_drop_frequent"] / "base_dataset.parquet", engine="pyarrow", index=False)

###### Tisztított zárójelentéseket tartalmazó adathalmazok létrehozása majd elmentése analízisre és tanításra (Creating and saving datasets containing cleaned discharge summaries for analysis and training)

In [ ]:
temp_dataset = dataset.copy()

In [ ]:
dataset["text"] = clean_text(dataset["text"].tolist(), "tfidf", nlp_core, BATCH_SIZE, N_PROCESS)
dataset.to_parquet(train_dirs["with_drop_frequent"] / "tfidf_cleaned_base_dataset.parquet", engine="pyarrow", index=False)

In [ ]:
temp_dataset["text"] = clean_text(temp_dataset["text"].tolist(), "modernbert")
temp_dataset.to_parquet(train_dirs["with_drop_frequent"] / "modernbert_cleaned_base_dataset.parquet", engine="pyarrow",index=False)

#### Top 50 ICD-10-CM kód (Top 50 ICD-10-CM code)

In [ ]:
top_50_code_dataset.drop(columns=["chapter"], inplace=True)

top_50_code_dataset_len = len(top_50_code_dataset)

code_counts = top_50_code_dataset["icd_code"].explode().value_counts()

print(f"Top 50 ICD-10-CM code counts:\n{code_counts.head(50)}\n")

top_50_code_set = set(code_counts.head(50).index)

top_50_code_dataset["icd_code"] = top_50_code_dataset["icd_code"].apply(
    lambda icd_code_list: [code for code in icd_code_list if code in top_50_code_set]
)

print(f"Number of removed code:\n{len(code_counts) - 50}\n")

top_50_code_dataset = top_50_code_dataset[top_50_code_dataset["icd_code"].str.len() > 0].reset_index(drop=True)

print(f"Rows dropped in 'top_50_code_base_dataset': {top_50_code_dataset_len-len(top_50_code_dataset)}")
print(f"Remaining rows in 'top_50_code_base_dataset': {len(top_50_code_dataset)}")

top_50_code_dataset.to_parquet(train_dirs["without_drop_top50"] / "base_dataset.parquet", engine="pyarrow", index=False)

In [ ]:
d_icd_diagnoses = d_icd_diagnoses[d_icd_diagnoses["icd_code"].isin(top_50_code_set)].drop(columns=["icd_version"])
d_icd_diagnoses.to_json(
    description_dir / "icd_code_descriptions.json", 
    orient="records", 
    indent=4, 
    force_ascii=False
)

###### Tisztított zárójelentéseket tartalmazó adathalmazok létrehozása majd elmentése analízisre és tanításra (Creating and saving datasets containing cleaned discharge summaries for analysis and training)

In [ ]:
dataset = top_50_code_dataset.copy()
dataset["text"] = clean_text(dataset["text"].tolist(),"tfidf", nlp_core, BATCH_SIZE, N_PROCESS)
dataset.to_parquet(train_dirs["without_drop_top50"] / "tfidf_cleaned_base_dataset.parquet", engine="pyarrow", index=False)

In [ ]:
dataset = top_50_code_dataset.copy()
dataset["text"] = clean_text(dataset["text"].tolist(),"modernbert")
dataset.to_parquet(train_dirs["without_drop_top50"] / "modernbert_cleaned_base_dataset.parquet", engine="pyarrow", index=False)

#### Záródiagnózisokat nem tartalmazó adathalmazok létrehozása majd elmentése analízisre és tanításra (Creating and saving datasets excluding discharge diagnoses for analysis and training)

In [ ]:
dataset = top_50_code_dataset.copy()
dataset["text"] = dataset["text"].progress_apply(
    lambda x: section_manager(
        x, 
        section_patterns, 
        "drop", 
        t_s_config_data["with_dropped_sections"]["sections_to_drop"]))
dataset.to_parquet(train_dirs["with_drop_top50"] / "base_dataset.parquet", engine="pyarrow", index=False)

###### Tisztított zárójelentéseket tartalmazó adathalmazok létrehozása majd elmentése analízisre és tanításra (Creating and saving datasets containing cleaned discharge summaries for analysis and training)

In [ ]:
temp_dataset = dataset.copy()

In [ ]:
dataset["text"] = clean_text(dataset["text"].tolist(), "tfidf", nlp_core, BATCH_SIZE, N_PROCESS)
dataset.to_parquet(train_dirs["with_drop_top50"] / "tfidf_cleaned_base_dataset.parquet", engine="pyarrow", index=False)

In [ ]:
temp_dataset["text"] = clean_text(temp_dataset["text"].tolist(), "modernbert")
temp_dataset.to_parquet(train_dirs["with_drop_top50"] / "modernbert_cleaned_base_dataset.parquet", engine="pyarrow", index=False)

### Csak a célszekciókat tartalmazó adathalmazok létrehozása majd elmentése analízisre és tanításra (Creating and saving datasets containing only target sections for analysis and training)


#### Gyakori ICD-10-CM chapter (Frequent ICD-10-CM chapter)

In [ ]:
dataset = frequent_chapter_dataset.copy()
dataset["text"] = dataset["text"].progress_apply(
    lambda x: section_manager(
        x, 
        section_patterns, 
        "keep", 
        t_s_config_data["without_dropped_sections"]["sections_to_keep"]))
dataset.to_parquet(train_dirs["without_drop_frequent"] / "t_s_dataset.parquet", engine="pyarrow", index=False)

###### Tisztított zárójelentéseket tartalmazó adathalmazok létrehozása majd elmentése analízisre és tanításra (Creating and saving datasets containing cleaned discharge summaries for analysis and training)

In [ ]:
temp_dataset = dataset.copy()

In [ ]:
dataset["text"] = clean_text(dataset["text"].tolist(), "tfidf", nlp_core, BATCH_SIZE, N_PROCESS)
dataset.to_parquet(train_dirs["without_drop_frequent"] / "tfidf_cleaned_t_s_dataset.parquet", engine="pyarrow", index=False)

In [ ]:
temp_dataset["text"] = clean_text(temp_dataset["text"].tolist(), "modernbert")
temp_dataset.to_parquet(train_dirs["without_drop_frequent"] / "modernbert_cleaned_t_s_dataset.parquet", engine="pyarrow", index=False)

#### Záródiagnózisokat nem tartalmazó adathalmazok létrehozása majd elmentése analízisre és tanításra (Creating and saving datasets excluding discharge diagnoses for analysis and training)

In [ ]:
dataset = frequent_chapter_dataset.copy()
dataset["text"] = dataset["text"].progress_apply(
    lambda x: section_manager(
        x, 
        section_patterns, 
        "keep", 
        t_s_config_data["with_dropped_sections"]["sections_to_keep"]))
dataset.to_parquet(train_dirs["with_drop_frequent"] / "t_s_dataset.parquet", engine="pyarrow", index=False)

###### Tisztított zárójelentéseket tartalmazó adathalmaz létrehozása majd elmentése analízisre és tanításra (Creating and saving dataset containing cleaned discharge summaries for analysis and training)

In [ ]:
temp_dataset = dataset.copy()

In [ ]:
dataset["text"] = clean_text(dataset["text"].tolist(), "tfidf", nlp_core, BATCH_SIZE, N_PROCESS)
dataset.to_parquet(train_dirs["with_drop_frequent"] / "tfidf_cleaned_t_s_dataset.parquet", engine="pyarrow", index=False)

In [ ]:
temp_dataset["text"] = clean_text(temp_dataset["text"].tolist(), "modernbert")
temp_dataset.to_parquet(train_dirs["with_drop_frequent"] / "modernbert_cleaned_t_s_dataset.parquet", engine="pyarrow", index=False)

#### Top 50 ICD-10-CM kód (Top 50 ICD-10-CM code)

##### Záródiagnózisokat tartalmazó adathalmazok létrehozása majd elmentése analízisre és tanításra (Creating and saving datasets including discharge diagnoses for analysis and training)

In [ ]:
dataset = top_50_code_dataset.copy()
dataset["text"] = dataset["text"].progress_apply(
    lambda x: section_manager(
        x, 
        section_patterns, 
        "keep", 
        t_s_config_data["without_dropped_sections"]["sections_to_keep"]))
dataset.to_parquet(train_dirs["without_drop_top50"] / "t_s_dataset.parquet", engine="pyarrow", index=False)

###### Tisztított zárójelentéseket tartalmazó adathalmazok létrehozása majd elmentése analízisre és tanításra (Creating and saving datasets containing cleaned discharge summaries for analysis and training)

In [ ]:
temp_dataset = dataset.copy()

In [ ]:
dataset["text"] = clean_text(dataset["text"].tolist(), "tfidf", nlp_core, BATCH_SIZE, N_PROCESS)
dataset.to_parquet(train_dirs["without_drop_top50"] / "tfidf_cleaned_t_s_dataset.parquet", engine="pyarrow", index=False)

In [ ]:
temp_dataset["text"] = clean_text(temp_dataset["text"].tolist(), "modernbert")
temp_dataset.to_parquet(train_dirs["without_drop_top50"] / "modernbert_cleaned_t_s_dataset.parquet", engine="pyarrow", index=False)

##### Záródiagnózisokat nem tartalmazó adathalmazok létrehozása majd elmentése analízisre és tanításra (Creating and saving datasets excluding discharge diagnoses for analysis and training)

In [ ]:
dataset = top_50_code_dataset.copy()
dataset["text"] = dataset["text"].progress_apply(
    lambda x: section_manager(
        x, 
        section_patterns, 
        "keep", 
        t_s_config_data["with_dropped_sections"]["sections_to_keep"]))
dataset.to_parquet(train_dirs["with_drop_top50"] / "t_s_dataset.parquet", engine="pyarrow", index=False)

###### Tisztított zárójelentéseket tartalmazó adathalmazok létrehozása majd elmentése analízisre és tanításra (Creating and saving datasets containing cleaned discharge summaries for analysis and training)

In [ ]:
temp_dataset = dataset.copy()

In [ ]:
dataset["text"] = clean_text(dataset["text"].tolist(), "tfidf", nlp_core, BATCH_SIZE, N_PROCESS)
dataset.to_parquet(train_dirs["with_drop_top50"] / "tfidf_cleaned_t_s_dataset.parquet", engine="pyarrow", index=False)

In [ ]:
temp_dataset["text"] = clean_text(temp_dataset["text"].tolist(), "modernbert")
temp_dataset.to_parquet(train_dirs["with_drop_top50"] / "modernbert_cleaned_t_s_dataset.parquet", engine="pyarrow", index=False)